<a href="https://www.kaggle.com/code/simplexcomplex/5-sratfold-cb-xbg-lbgm-ensemble-lb-0-94993?scriptVersionId=331828584" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Predicting Student Health Risk | Kaggle Playground S6E7

This notebook builds a reproducible tabular machine-learning pipeline for the **Playground Series - Season 6, Episode 7** competition.

**Goal:** predict `health_condition` with one of three labels: `at-risk`, `unhealthy`, or `fit`.

**Evaluation metric:** balanced accuracy. This metric averages recall across classes, so the pipeline uses class-balanced training instead of optimizing ordinary accuracy on the majority class.

The final submission is a weighted probability ensemble of LightGBM, CatBoost, and XGBoost trained with stratified cross-validation.

In [1]:
# Reproducibility and notebook configuration
from pathlib import Path
import os
import random
import warnings

SEED = 42
N_SPLITS = 5

# Set this to True for a quick smoke test while editing the notebook.
# Keep False for the public Kaggle run and final submission.
DEBUG = False
DEBUG_ROWS = 80_000

# Model blending weights. They are intentionally conservative and sum to 1.
MODEL_WEIGHTS = {
    "lightgbm": 0.25,
    "catboost": 0.35,
    "xgboost": 0.40,
}

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
random.seed(SEED)
warnings.filterwarnings("ignore")

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing import LabelEncoder

import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from lightgbm import LGBMClassifier, early_stopping as lgb_early_stopping, log_evaluation as lgb_log_evaluation
from catboost import CatBoostClassifier, Pool
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("lightgbm:", lgb.__version__)
print("xgboost:", xgb.__version__)
print("catboost:", cb.__version__)

numpy: 2.0.2
pandas: 2.3.3
lightgbm: 4.6.0
xgboost: 3.2.0
catboost: 1.2.10


## Data Loading

The path logic works both on Kaggle and in a local checkout of the competition files.

In [3]:
def find_data_dir() -> Path:
    candidates = [
        Path("/kaggle/input/competitions/playground-series-s6e7"),
        Path("playground-series-s6e7"),
    ]
    for path in candidates:
        if (path / "train.csv").exists() and (path / "test.csv").exists():
            return path
    raise FileNotFoundError("Could not find train.csv and test.csv. Add the Kaggle dataset or update the path list.")

DATA_DIR = find_data_dir()
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

if DEBUG:
    train = train.sample(DEBUG_ROWS, random_state=SEED).sort_values("id").reset_index(drop=True)

print("DATA_DIR:", DATA_DIR)
print("train:", train.shape)
print("test:", test.shape)
print("sample_submission:", sample_submission.shape)
train.head()

DATA_DIR: /kaggle/input/competitions/playground-series-s6e7
train: (690088, 15)
test: (295753, 14)
sample_submission: (295753, 2)


,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


In [4]:
target = "health_condition"
id_col = "id"

print("Target distribution")
display(train[target].value_counts())

print("Target distribution (%)")
display((train[target].value_counts(normalize=True) * 100).round(2))

print("Missing values")
missing = train.isna().sum().sort_values(ascending=False)
display(missing[missing > 0])

Target distribution


health_condition
at-risk      592561
unhealthy     57724
fit           39803
Name: count, dtype: int64

Target distribution (%)


health_condition
at-risk      85.87
unhealthy     8.36
fit           5.77
Name: proportion, dtype: float64

Missing values


stress_level               82811
sleep_duration             75999
sleep_quality              58331
calorie_expenditure        52853
water_intake               43477
physical_activity_level    36621
smoking_alcohol            28582
gender                     21373
step_count                 13916
bmi                        13898
heart_rate                  7833
diet_type                   6901
exercise_duration           6901
dtype: int64

## Feature Preparation

All models use the same feature columns. Categorical features are passed as categorical data where the library supports it. Missing categorical values are made explicit as `__missing__`; missing numeric values are left as `NaN` because all three tree boosters can handle them.

In [5]:
feature_cols = [c for c in train.columns if c not in [id_col, target]]
categorical_cols = train[feature_cols].select_dtypes(include=["object"]).columns.tolist()
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

print("Numeric features:", numeric_cols)
print("Categorical features:", categorical_cols)

TARGET_CLASSES = np.array(["at-risk", "fit", "unhealthy"])
label_encoder = LabelEncoder()
label_encoder.fit(TARGET_CLASSES)
print("Target class order:", list(label_encoder.classes_))

Numeric features: ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']
Categorical features: ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']
Target class order: [np.str_('at-risk'), np.str_('fit'), np.str_('unhealthy')]


In [6]:
def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    # Return model-ready features with stable categorical dtypes.
    X = df[feature_cols].copy()
    for col in categorical_cols:
        X[col] = X[col].astype("object").fillna("__missing__").astype("category")
    return X

X = prepare_features(train)
X_test = prepare_features(test)
y = train[target].copy()
y_encoded = label_encoder.transform(y)

test_ids = test[id_col].copy()

print(X.shape, X_test.shape)
X.head()

(690088, 13) (295753, 13)


,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,__missing__,average,sedentary,__missing__,male


## Model Definitions

Balanced accuracy rewards recall for every class equally. The training setup therefore compensates for imbalance:

- **LightGBM:** `class_weight="balanced"`
- **CatBoost:** `auto_class_weights="Balanced"`
- **XGBoost:** per-row balanced sample weights

Each model uses a fixed random seed. Cross-validation is stratified so every fold keeps the same class mix.

In [7]:
def make_lightgbm(seed: int) -> LGBMClassifier:
    return LGBMClassifier(
        objective="multiclass",
        n_estimators=2500,
        learning_rate=0.035,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=45,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=0.50,
        class_weight="balanced",
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    )


def make_catboost(seed: int) -> CatBoostClassifier:
    return CatBoostClassifier(
        loss_function="MultiClass",
        iterations=2500,
        learning_rate=0.035,
        depth=8,
        l2_leaf_reg=6.0,
        random_strength=0.5,
        auto_class_weights="Balanced",
        bootstrap_type="Bernoulli",
        subsample=0.85,
        random_seed=seed,
        allow_writing_files=False,
        verbose=False,
        thread_count=-1,
    )


def make_xgboost(seed: int) -> XGBClassifier:
    return XGBClassifier(
        objective="multi:softprob",
        num_class=len(TARGET_CLASSES),
        n_estimators=1600,
        learning_rate=0.035,
        max_depth=6,
        min_child_weight=8,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=1.50,
        tree_method="hist",
        enable_categorical=True,
        eval_metric="mlogloss",
        early_stopping_rounds=100,
        random_state=seed,
        n_jobs=-1,
    )


def align_proba(proba: np.ndarray, model_classes) -> np.ndarray:
    # Align model probability columns to TARGET_CLASSES.
    aligned = np.zeros((proba.shape[0], len(TARGET_CLASSES)), dtype=np.float64)
    model_classes = np.array(model_classes).astype(str)
    for i, cls in enumerate(model_classes):
        aligned[:, np.where(TARGET_CLASSES == cls)[0][0]] = proba[:, i]
    return aligned


def balanced_score_from_proba(y_true, proba: np.ndarray) -> float:
    pred = TARGET_CLASSES[np.argmax(proba, axis=1)]
    return balanced_accuracy_score(y_true, pred)

## Cross-Validation

This cell trains the full ensemble. It prints fold scores for every model and for the blended OOF prediction. The OOF score is the best local estimate of public leaderboard behavior.

In [8]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_pred = {name: np.zeros((len(X), len(TARGET_CLASSES)), dtype=np.float64) for name in MODEL_WEIGHTS}
test_pred = {name: np.zeros((len(X_test), len(TARGET_CLASSES)), dtype=np.float64) for name in MODEL_WEIGHTS}
fold_rows = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n===== Fold {fold}/{N_SPLITS} =====")
    seed = SEED + fold
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    y_train_encoded = label_encoder.transform(y_train)
    y_valid_encoded = label_encoder.transform(y_valid)
    
    # LightGBM
    lgbm = make_lightgbm(seed)
    lgbm.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        categorical_feature=categorical_cols,
        callbacks=[lgb_early_stopping(100, verbose=False), lgb_log_evaluation(0)],
    )
    lgb_valid = align_proba(lgbm.predict_proba(X_valid), lgbm.classes_)
    lgb_test = align_proba(lgbm.predict_proba(X_test), lgbm.classes_)
    oof_pred["lightgbm"][valid_idx] = lgb_valid
    test_pred["lightgbm"] += lgb_test / N_SPLITS
    lgb_score = balanced_score_from_proba(y_valid, lgb_valid)
    print(f"LightGBM balanced accuracy: {lgb_score:.6f}")
    
    # CatBoost
    cat_train = Pool(X_train, y_train, cat_features=categorical_cols)
    cat_valid = Pool(X_valid, y_valid, cat_features=categorical_cols)
    cat_test_pool = Pool(X_test, cat_features=categorical_cols)
    cat = make_catboost(seed)
    cat.fit(cat_train, eval_set=cat_valid, early_stopping_rounds=100, use_best_model=True)
    cat_valid_proba = align_proba(cat.predict_proba(cat_valid), cat.classes_)
    cat_test_proba = align_proba(cat.predict_proba(cat_test_pool), cat.classes_)
    oof_pred["catboost"][valid_idx] = cat_valid_proba
    test_pred["catboost"] += cat_test_proba / N_SPLITS
    cat_score = balanced_score_from_proba(y_valid, cat_valid_proba)
    print(f"CatBoost balanced accuracy: {cat_score:.6f}")
    
    # XGBoost
    xgb = make_xgboost(seed)
    sample_weight = compute_sample_weight(class_weight="balanced", y=y_train_encoded)
    valid_weight = compute_sample_weight(class_weight="balanced", y=y_valid_encoded)
    xgb.fit(
        X_train,
        y_train_encoded,
        sample_weight=sample_weight,
        eval_set=[(X_valid, y_valid_encoded)],
        sample_weight_eval_set=[valid_weight],
        verbose=False,
    )
    xgb_valid = align_proba(xgb.predict_proba(X_valid), TARGET_CLASSES)
    xgb_test = align_proba(xgb.predict_proba(X_test), TARGET_CLASSES)
    oof_pred["xgboost"][valid_idx] = xgb_valid
    test_pred["xgboost"] += xgb_test / N_SPLITS
    xgb_score = balanced_score_from_proba(y_valid, xgb_valid)
    print(f"XGBoost balanced accuracy: {xgb_score:.6f}")
    
    blend_valid = sum(MODEL_WEIGHTS[name] * oof_pred[name][valid_idx] for name in MODEL_WEIGHTS)
    blend_score = balanced_score_from_proba(y_valid, blend_valid)
    print(f"Blend balanced accuracy: {blend_score:.6f}")
    
    fold_rows.append({
        "fold": fold,
        "lightgbm": lgb_score,
        "catboost": cat_score,
        "xgboost": xgb_score,
        "blend": blend_score,
    })

cv_results = pd.DataFrame(fold_rows)
display(cv_results)
print("\nMean CV scores")
display(cv_results.drop(columns="fold").agg(["mean", "std"]).T.sort_values("mean", ascending=False))


===== Fold 1/5 =====
LightGBM balanced accuracy: 0.939700
CatBoost balanced accuracy: 0.949894
XGBoost balanced accuracy: 0.950323
Blend balanced accuracy: 0.950201

===== Fold 2/5 =====
LightGBM balanced accuracy: 0.940027
CatBoost balanced accuracy: 0.951029
XGBoost balanced accuracy: 0.951918
Blend balanced accuracy: 0.951427

===== Fold 3/5 =====
LightGBM balanced accuracy: 0.941030
CatBoost balanced accuracy: 0.949090
XGBoost balanced accuracy: 0.949179
Blend balanced accuracy: 0.949154

===== Fold 4/5 =====
LightGBM balanced accuracy: 0.938146
CatBoost balanced accuracy: 0.949015
XGBoost balanced accuracy: 0.949577
Blend balanced accuracy: 0.948875

===== Fold 5/5 =====
LightGBM balanced accuracy: 0.937722
CatBoost balanced accuracy: 0.947945
XGBoost balanced accuracy: 0.948343
Blend balanced accuracy: 0.947759


,fold,lightgbm,catboost,xgboost,blend
0,1,0.939700,0.949894,0.950323,0.950201
1,2,0.940027,0.951029,0.951918,0.951427
2,3,0.941030,0.949090,0.949179,0.949154
3,4,0.938146,0.949015,0.949577,0.948875
4,5,0.937722,0.947945,0.948343,0.947759



Mean CV scores


,mean,std
xgboost,0.949868,0.001350
blend,0.949483,0.001391
catboost,0.949395,0.001147
lightgbm,0.939325,0.001369


In [9]:
blend_oof = sum(MODEL_WEIGHTS[name] * oof_pred[name] for name in MODEL_WEIGHTS)
oof_labels = TARGET_CLASSES[np.argmax(blend_oof, axis=1)]
oof_score = balanced_accuracy_score(y, oof_labels)

print(f"OOF blended balanced accuracy: {oof_score:.6f}")
print("\nClassification report")
print(classification_report(y, oof_labels, digits=4))

cm = pd.DataFrame(
    confusion_matrix(y, oof_labels, labels=TARGET_CLASSES),
    index=[f"true_{c}" for c in TARGET_CLASSES],
    columns=[f"pred_{c}" for c in TARGET_CLASSES],
)
display(cm)

OOF blended balanced accuracy: 0.949483

Classification report
              precision    recall  f1-score   support

     at-risk     0.9929    0.9404    0.9660    592561
         fit     0.7392    0.9486    0.8309     39803
   unhealthy     0.7119    0.9594    0.8173     57724

    accuracy                         0.9425    690088
   macro avg     0.8147    0.9495    0.8714    690088
weighted avg     0.9548    0.9425    0.9458    690088



,pred_at-risk,pred_fit,pred_unhealthy
true_at-risk,557259,13094,22208
true_fit,1844,37757,202
true_unhealthy,2114,228,55382


## Submission

The final prediction is the average of each model's fold-averaged test probabilities, blended with `MODEL_WEIGHTS`.

In [10]:
blend_test = sum(MODEL_WEIGHTS[name] * test_pred[name] for name in MODEL_WEIGHTS)
test_labels = TARGET_CLASSES[np.argmax(blend_test, axis=1)]

submission = pd.DataFrame({
    id_col: test_ids,
    target: test_labels,
})

# Sanity checks against the sample submission format
assert list(submission.columns) == list(sample_submission.columns)
assert len(submission) == len(sample_submission)
assert submission[id_col].equals(sample_submission[id_col])
assert set(submission[target]).issubset(set(TARGET_CLASSES))

submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
display(submission.head())
display(submission[target].value_counts())

Saved: /kaggle/working/submission.csv


,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


health_condition
at-risk      240931
unhealthy     33193
fit           21629
Name: count, dtype: int64

## Notes

- The competition metric is balanced accuracy, so the model intentionally favors recall balance over raw accuracy.
- The dataset has substantial class imbalance; predicting only `at-risk` gives high ordinary accuracy but only `0.3333` balanced accuracy.
- For faster iteration, set `DEBUG = True` near the top. Keep it `False` for the final Kaggle submission run.